# Task 2: Predict Future Stock Prices (Short-Term)

**Objective:** Use historical stock data to predict the next day's closing price.

**Dataset:** Yahoo Finance via `yfinance` library (Apple Inc. – AAPL)

**Models:** Linear Regression & Random Forest Regressor

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 5)
print("Libraries loaded.")

## 2. Fetch Stock Data

In [ ]:
import yfinance as yf

TICKER = "AAPL"
START  = "2020-01-01"
END    = "2024-12-31"

print(f"Downloading {TICKER} data from {START} to {END}...")
df = yf.download(TICKER, start=START, end=END, progress=False)
print(f"Shape: {df.shape}")
df.head()

## 3. Data Preprocessing & Feature Engineering

In [ ]:
# Flatten multi-level columns if present
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Select relevant features
df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
df.dropna(inplace=True)

# ── Feature Engineering ──────────────────────────────────
df['MA_5']       = df['Close'].rolling(5).mean()   # 5-day moving average
df['MA_20']      = df['Close'].rolling(20).mean()  # 20-day moving average
df['Daily_Range']= df['High'] - df['Low']          # Day's price range
df['Pct_Change'] = df['Close'].pct_change()        # Day-over-day % change
df['Lag_1']      = df['Close'].shift(1)            # Previous close (lag)
df['Lag_2']      = df['Close'].shift(2)
df['Lag_3']      = df['Close'].shift(3)

# Target: next day's close
df['Target'] = df['Close'].shift(-1)

# Drop rows with NaN from rolling/shift
df.dropna(inplace=True)
print(f"Final dataset shape: {df.shape}")
df.tail()

## 4. Train / Test Split

In [ ]:
FEATURES = ['Open', 'High', 'Low', 'Volume', 'MA_5', 'MA_20',
            'Daily_Range', 'Pct_Change', 'Lag_1', 'Lag_2', 'Lag_3']
TARGET = 'Target'

X = df[FEATURES]
y = df[TARGET]

# Time-series split (no shuffle!)
split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]
dates_test = df.index[split:]

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")

## 5. Model Training

In [ ]:
# ── Linear Regression ────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
lr_preds = lr.predict(X_test_sc)

# ── Random Forest ─────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=200, max_depth=10,
                            random_state=42, n_jobs=-1)
rf.fit(X_train_sc, y_train)
rf_preds = rf.predict(X_test_sc)

print("Both models trained successfully.")

## 6. Model Evaluation

In [ ]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"{'─'*40}")
    print(f"  {name}")
    print(f"  MAE  : ${mae:.2f}")
    print(f"  RMSE : ${rmse:.2f}")
    print(f"  R²   : {r2:.4f}")

evaluate("Linear Regression", y_test, lr_preds)
evaluate("Random Forest",     y_test, rf_preds)

## 7. Visualization – Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

for ax, preds, name, color in zip(
        axes,
        [lr_preds, rf_preds],
        ["Linear Regression", "Random Forest"],
        ["#E74C3C", "#2ECC71"]):

    ax.plot(dates_test, y_test.values, label='Actual Close',
            color='#3498DB', linewidth=1.5, alpha=0.9)
    ax.plot(dates_test, preds, label=f'{name} Predicted',
            color=color, linewidth=1.2, linestyle='--', alpha=0.85)
    ax.set_title(f'AAPL – {name}: Actual vs Predicted Close Price',
                 fontsize=13, fontweight='bold')
    ax.set_ylabel('Price (USD)')
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('stock_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved.")

### 7.1 Feature Importance – Random Forest

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
colors = ['#F39C12' if v > 0.1 else '#3498DB' for v in importances.values]
importances.plot(kind='barh', color=colors, edgecolor='white')
plt.title('Random Forest – Feature Importances', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance_stocks.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Key Insights

- **Lag features** (previous close prices) are the strongest predictors — the most recent close has the highest importance.
- **Random Forest** outperforms Linear Regression, achieving a higher R² and lower MAE/RMSE on test data.
- **Moving averages** (MA_5, MA_20) provide useful trend context but are slightly redundant with lag features.
- Stock prediction is inherently noisy; even strong R² values don't guarantee profitable trading strategies without additional signals.
- The model generalizes well on the 2023-2024 period despite volatile market conditions.